## The Extended Kalman Filter 

This cell implements the Extended Kalman Filter for the $\mathcal{VKSSM}$.

At each time step, the current filtered mean and covariance are first propagated through the nonlinear transition map to obtain a predicted state. The transition is then linearised about the current estimate using the Jacobian of choice (either the Fixed interaction graph or the Smooth Exponential surrogate).

The observation consists of noisy agent positions only. The innovation is therefore the difference between the observed positions and the predicted positions. Since positions lie on a periodic box, this innovation is wrapped so that the correction uses the shortest displacement on the torus.

The filter then performs the standard measurement update using the Kalman gain. After the update, positions are wrapped back into the box and headings are wrapped back into $[-\pi,\pi)$. The covariance is updated to improve numerical stability.

The second function applies this recursion to a full simulated trajectory. It generates noisy position observations from the true positions, constructs an initial perturbed estimate of the state, specifies the initial covariance, and then runs the EKF forward in time. The output is the observation sequence together with the filtered state means.

In [ ]:
# EKF

def ekf_step_vk(m, C, y_obs, params, Q, R, H):
    """
    One EKF step for the smooth VK state space model.
    """
    # Predict the next state using the nonlinear transition map,
    # then linearise that map about the current estimate.
    m_pred = vk_state_transition(m, params)

    # Calculate your Jacobian of choice by running either the Fixed Interaction block or the Gaussian Interaction block.
    F = vk_jacobian_F_approx(m, params)
    C_pred = F @ C @ F.T + Q

    # Predicted observation error.
    innovation = y_obs - observation_h(m_pred)

    # Position observations live on a periodic box
    innovation[0::2] = innovation[0::2] - params.L * np.round(innovation[0::2] / params.L)
    innovation[1::2] = innovation[1::2] - params.L * np.round(innovation[1::2] / params.L)

    # Standard EKF measurement update.
    S = H @ C_pred @ H.T + R
    K = C_pred @ H.T @ np.linalg.inv(S)
    m_new = m_pred + K @ innovation

    # After the linear update, wrap positions and headings back to
    # the physical state space.
    pos_new, theta_new = unpack_state(m_new)
    pos_new = wrap_box(pos_new, params.L)
    theta_new = wrap_angle(theta_new)
    m_new = pack_state(pos_new, theta_new)

    # Covariance update with Joseph form for numerical stability.
    I = np.eye(len(m))
    C_new = (I - K @ H) @ C_pred @ (I - K @ H).T + K @ R @ K.T
    C_new = 0.5 * (C_new + C_new.T)

    return m_new, C_new


def run_ekf_vk_from_truth(pos, theta, params, obs_std=0.30, seed=123):
    """
    Run the EKF using noisy position observations generated from a true trajectory.

    Inputs
    ------
    pos   : (T+1, N, 2)
    theta : (T+1, N)

    Returns
    -------
    observations : (T+1, 2N)
    filt_means   : (T+1, 3N)
    """
    T_plus_1, N, _ = pos.shape

    # Build noisy position observations from the simulated trajectory.
    observations = generate_noisy_observations_from_pos(pos, obs_std=obs_std, seed=seed)

    rng = np.random.default_rng(seed + 1)

    # Initialise the filter near the true initial state, with small
    # perturbations in position and heading.
    pos0_est = wrap_box(pos[0] + 0.15 * rng.standard_normal(size=(N, 2)), params.L)
    theta0_est = wrap_angle(theta[0] + 0.35 * rng.standard_normal(size=N))
    m0 = pack_state(pos0_est, theta0_est)

    state_dim = 3 * N

    # Initial covariance: separate variances for x, y and theta.
    C0 = np.eye(state_dim)
    for i in range(N):
        C0[3 * i, 3 * i] = 0.20**2
        C0[3 * i + 1, 3 * i + 1] = 0.20**2
        C0[3 * i + 2, 3 * i + 2] = 0.50**2

    # Observation model: positions only.
    H = observation_matrix_H(N)
    R = (obs_std**2) * np.eye(2 * N)

    # Process noise is placed on the heading components only.
    Q = np.zeros((state_dim, state_dim))
    for i in range(N):
        Q[3 * i + 2, 3 * i + 2] = (params.sigma**2) * params.dt

    # Store filtered means and covariances over time.
    filt_means = np.zeros((T_plus_1, state_dim))
    filt_covs = np.zeros((T_plus_1, state_dim, state_dim))
    filt_means[0] = m0
    filt_covs[0] = C0

    # At time t, use observation y_{t+1} to update the prediction from state t.
    for t in range(T_plus_1 - 1):
        m_new, C_new = ekf_step_vk(
            m=filt_means[t],
            C=filt_covs[t],
            y_obs=observations[t + 1],
            params=params,
            Q=Q,
            R=R,
            H=H
        )
        filt_means[t + 1] = m_new
        filt_covs[t + 1] = C_new

    return observations, filt_means

In [ ]:
observations, filt_means = run_ekf_vk_from_truth(
    pos=pos,
    theta=theta,
    params=p,
    obs_std=0.30,
    seed=123
)